# UA RoBERTa Fine-tuning на UNLP 2025

In [ ]:
!pip install transformers datasets torch accelerate sentencepiece

In [ ]:
from datasets import load_dataset
ds = load_dataset("lasr-unlp/unlp-2025-shared-task", trust_remote_code=True)
df = ds["train"].to_pandas()
print(df["label"].value_counts())

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("youscan/ukr-roberta-base")
def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)
tokenized_ds = ds.map(tokenize_fn, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
model = AutoModelForSequenceClassification.from_pretrained("youscan/ukr-roberta-base", num_labels=2)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    learning_rate=2e-5,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"]
)
trainer.train()

In [ ]:
import os
os.makedirs("artifacts/ua_roberta_model", exist_ok=True)
model.save_pretrained("artifacts/ua_roberta_model")
tokenizer.save_pretrained("artifacts/ua_roberta_model")
print("Model saved to artifacts/ua_roberta_model")

## Висновки
Модель збережено. Вона може бути використана як `UkrainianClassifier` у нашому пайплайні.